# Fine-tune ReProver (ByT5-small) on Mathlib Calculus Tactics

**Run on:** Google Colab with GPU runtime  
**Runtime → Change runtime type → T4 GPU** (free) or **A100** (Colab Pro)

This notebook:
1. Installs dependencies
2. Downloads the Mathlib calculus tactic dataset (auto-filtered)
3. Downloads the pretrained ReProver ByT5-small checkpoint
4. Fine-tunes for 10 epochs (~2–8 hours depending on GPU)
5. Evaluates top-k exact-match accuracy on the held-out test set
6. Saves the fine-tuned model to Google Drive


In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'No GPU found — change runtime to GPU!')

In [ ]:
%%capture
!pip install transformers datasets huggingface_hub sentencepiece accelerate

In [ ]:
# ── 1. Download and filter Mathlib calculus tactic data ──────────────────────
import json
from pathlib import Path
from datasets import load_dataset

DATASET_ID = 'cat-searcher/leandojo-benchmark-4-random'

def is_calculus(ex):
    return 'Calculus' in ex['file_path']

def save_jsonl(records, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        for r in records:
            f.write(json.dumps(r) + '\n')
    print(f'Saved {len(records)} examples → {path}')

for split_name, hf_name in [('train', 'train'), ('val', 'validation'), ('test', 'test')]:
    ds = load_dataset(DATASET_ID, split=hf_name)
    records = [
        {'state': ex['state'], 'tactic': ex['tactic'],
         'full_name': ex['full_name'], 'file_path': ex['file_path']}
        for ex in ds if is_calculus(ex)
    ]
    save_jsonl(records, f'data/calculus/{split_name}.jsonl')

In [ ]:
# ── 2. Download pretrained ReProver checkpoint ────────────────────────────────
from huggingface_hub import snapshot_download

MODEL_ID  = 'kaiyuy/leandojo-lean4-tacgen-byt5-small'
MODEL_DIR = 'models/pretrained/leandojo-lean4-tacgen-byt5-small'

snapshot_download(repo_id=MODEL_ID, local_dir=MODEL_DIR)
print('Model downloaded to', MODEL_DIR)

In [ ]:
# ── 3. Fine-tune ──────────────────────────────────────────────────────────────
import json
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)

BASE_MODEL  = 'models/pretrained/leandojo-lean4-tacgen-byt5-small'
OUTPUT_DIR  = 'models/finetuned/calculus'
EPOCHS      = 10
BATCH_SIZE  = 16     # increase to 32 on A100
GRAD_ACCUM  = 1
LR          = 3e-4
MAX_IN_LEN  = 1024
MAX_OUT_LEN = 256

class TacticDataset(Dataset):
    def __init__(self, path, tokenizer):
        self.tokenizer = tokenizer
        self.examples = []
        with open(path) as f:
            for line in f:
                r = json.loads(line)
                s, t = r['state'].strip(), r['tactic'].strip()
                if s and t:
                    self.examples.append((s, t))
        print(f'Loaded {len(self.examples)} examples from {path}')

    def __len__(self): return len(self.examples)

    def __getitem__(self, idx):
        state, tactic = self.examples[idx]
        inp = self.tokenizer(state, max_length=MAX_IN_LEN, truncation=True, padding=False)
        lbl = self.tokenizer(text_target=tactic, max_length=MAX_OUT_LEN, truncation=True, padding=False)
        inp['labels'] = lbl['input_ids']
        return inp

def make_metrics(tokenizer):
    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        if isinstance(predictions, tuple): predictions = predictions[0]
        preds  = tokenizer.batch_decode(predictions, skip_special_tokens=True)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        labs   = tokenizer.batch_decode(labels, skip_special_tokens=True)
        em = sum(p.strip() == l.strip() for p, l in zip(preds, labs)) / max(len(labs), 1)
        return {'exact_match': em}
    return compute_metrics

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model     = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

train_ds = TacticDataset('data/calculus/train.jsonl', tokenizer)
val_ds   = TacticDataset('data/calculus/val.jsonl',   tokenizer)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model,
                                  label_pad_token_id=-100, pad_to_multiple_of=8)

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=200,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    predict_with_generate=True,
    generation_max_length=256,
    generation_num_beams=4,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='exact_match',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=make_metrics(tokenizer),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

In [ ]:
# ── 4. Evaluate top-k exact match on test set ─────────────────────────────────
import json, random
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
K = 10

tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
model     = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR).to(DEVICE)
model.eval()

test_examples = []
with open('data/calculus/test.jsonl') as f:
    for line in f:
        r = json.loads(line)
        test_examples.append((r['state'].strip(), r['tactic'].strip()))

top1 = topk = 0
for state, true_tactic in test_examples:
    inputs = tokenizer(state, return_tensors='pt',
                       max_length=1024, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs, num_beams=K, num_return_sequences=K,
            max_new_tokens=256, early_stopping=True,
        )
    preds = [tokenizer.decode(seq, skip_special_tokens=True).strip() for seq in out]
    if preds and preds[0] == true_tactic: top1 += 1
    if true_tactic in preds:              topk += 1

n = len(test_examples)
print(f'Test set: {n} examples')
print(f'Top-1  exact match: {top1/n:.3f} ({top1}/{n})')
print(f'Top-{K} exact match: {topk/n:.3f} ({topk}/{n})')

with open(f'{OUTPUT_DIR}/test_metrics.json', 'w') as f:
    json.dump({'n': n, 'top1': top1/n, f'top{K}': topk/n}, f, indent=2)

In [ ]:
# ── 5. Save model to Google Drive ─────────────────────────────────────────────
# Uncomment and run this cell to persist the model across Colab sessions

# from google.colab import drive
# import shutil
# drive.mount('/content/drive')
# shutil.copytree('models/finetuned/calculus',
#                 '/content/drive/MyDrive/REU/models/finetuned/calculus',
#                 dirs_exist_ok=True)
# print('Model saved to Google Drive')
print('Uncomment the lines above to save to Google Drive')